In [ ]:
!pip install sentence-transformers chromadb groq --q

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")
print("ready to build a RAG system")

All libraries imported successfully
ready to build a RAG system


In [ ]:
GROQ_API_KEY="Groq_Api_Key"
os.environ["GROQ_API_KEY"]="GROQ_API_KEY"
groq_client=Groq(api_key=GROQ_API_KEY)
print("Groq api client initialised.")


Groq api client initialised.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving college_notes.csv to college_notes.csv


In [ ]:
df=pd.read_csv('college_notes.csv')
print("shape:",df.shape)
print("column",df.columns.tolist())
print("first 3 rows:")
print(df.head(3))

shape: (15, 4)
column ['note_id', 'subject', 'topic', 'content']
first 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [ ]:
print("subject int the dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content(number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

subject int the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"total chunks prepared: {len(documents)}")
print(f"first document ID: {ids[14]}")
print(f"first metadata: {metadatas[0]}")
print(f"first 100 chars of doc:{documents[0][:100]}...")

total chunks prepared: 15
first document ID: note_N015
first metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
first 100 chars of doc:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("subsequent runs will be faster as the model is cached")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("embedding model initialised")
test_embedding=embedding_model.encode("this is a test sentence")
print(f"first 5 values of test embedding:{test_embedding[:5]}")
#

subsequent runs will be faster as the model is cached


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model initialised
first 5 values of test embedding:[0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("chromadb client created")
print(f"Collection name:college_notes_rag)")
print(f"Documents in collection so far:{collection.count()}")

chromadb client created
Collection name:college_notes_rag)
Documents in collection so far:0


In [ ]:
print("generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print("embeddings generated")
print(f"\nEmbedding matrix shape:{embeddings.shape}")
embeddings_list=embeddings.tolist()
collection.add(documents=documents,
               embeddings=embeddings_list,
               metadatas=metadatas,
               ids=ids)
print(f"\nDocuments in collection:{collection.count()}")
print(f"Total documents in collection:{collection.count()}")


generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings generated

Embedding matrix shape:(15, 384)

Documents in collection:15
Total documents in collection:15


In [ ]:
def retrieve (question,top_k=3):

  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results
print("function determine successfully")

function determine successfully


In [ ]:
t_q="what is ETL and how does it works"
results=retrieve(t_q,top_k=3)
print("\n top retrieved chunks")
for i, (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nresult:{i+1}:")
  print(f"subject:{meta['subject']}")
  print(f"topic:{meta['topic']}")
  print(f"distance:{dist:.4f}")
  print(f"content:{doc[:100]}...")




 top retrieved chunks

result:1:
subject:Data Engineering
topic:ETL Pipelines
distance:0.2761
content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...

result:2:
subject:Data Engineering
topic:APIs and Data Collection
distance:1.3964
content:An API or Application Programming Interface allows two software applications to talk to each other. ...

result:3:
subject:Generative AI
topic:Retrieval Augmented Generation
distance:1.5003
content:RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant docu...


In [ ]:
def build_content(results):
  context_parts=[]
  for i,(doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[Source{i+1}:{meta['subject']}-{meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context_str="\n\n---\n\n".join(context_parts)
  return context_str
context=build_content(results)
print("built context string from retrieved chunks:")
print(context[:500] +"...")
print(f"\nContext length:{len(context)}")


built context string from retrieved chunks:
[Source1:Data Engineering-ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

[Source2:Data Engineering-APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather ...

Context length:876


In [ ]:
def generate_rag_answer(question,context):
    system_prompt="""
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible."""

    user_prompt = f"""Context from Knowledge Base:

    {context}

    Student's Question: {question}
    Please answer the question based only on the context provided above."""

    response=groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}
    ],
    temperature=0.1,
    max_tokens=500
    )
    answer=response.choices[0].message.content

    return answer
print("generate rag defined")





generate rag defined


In [ ]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question:{question}")
    print("step 1")
  results=retrieve(question,top_k=top_k)
  if verbose:
    print(f"retrieved {top_k} chunks from the knowledge base:")
    for i, meta in enumerate(results['metadatas'][0]):
        print(f" {i+1}. {meta['subject']}-{meta['topic']}")
    print("\n step 2")
  context=build_content(results)
  if verbose:
    print(f"context built({len(context)} characters)")
    print("\n step 3:")
  answer=generate_rag_answer(question,context)
  if verbose:
    print("\n" + "=" *60)
    print("Answer:")
    print(answer)
  return answer
print("complete rag pipeline")
print("Function:ask college assistant(question,top_k=3)")

complete rag pipeline
Function:ask college assistant(question,top_k=3)


In [ ]:
question_1="what is ETL and what are its main three main stages?"
answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)


Question:what is ETL and what are its main three main stages?
step 1
retrieved 3 chunks from the knowledge base:
 1. Data Engineering-ETL Pipelines
 2. Generative AI-Prompt Engineering
 3. Generative AI-Retrieval Augmented Generation

 step 2
context built(918 characters)

 step 3:

Answer:
ETL stands for Extract Transform Load. 

Its main three stages are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.
